# 01 · Landing zone profiling
New files for 15–21 Oct arrived in `de-lab/data/landing/`. Before loading anything, profile them. **Days 4–5.** Always read as text first (`all_varchar=true`) so DuckDB does not silently guess types.

In [1]:
import pathlib, duckdb
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "warehouse.duckdb").exists())
wh = duckdb.connect(str(ROOT / "warehouse.duckdb"), read_only=True)   # production: ALWAYS read-only
def q(sql, con=None):
    return (con or wh).sql(sql)
LANDING = ROOT / 'de-lab/data/landing'
sorted(p.name for p in (LANDING / 'orders').glob('*.csv'))

['orders_2026-10-15.csv',
 'orders_2026-10-16.csv',
 'orders_2026-10-17.csv',
 'orders_2026-10-18.csv',
 'orders_2026-10-19.csv',
 'orders_2026-10-21.csv']

## Header of every file: spot schema drift

In [2]:
for p in sorted((LANDING / 'orders').glob('*.csv')):
    cols = [c[0] for c in wh.sql(f"DESCRIBE SELECT * FROM read_csv('{p}', all_varchar=true)").fetchall()]
    print(p.name, cols)

orders_2026-10-15.csv ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'status', 'amount', 'promo_value', 'payment_method', 'cancel_reason']
orders_2026-10-16.csv ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'status', 'amount', 'promo_value', 'payment_method', 'cancel_reason']
orders_2026-10-17.csv ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'status', 'amount', 'promo_value', 'payment_method', 'cancel_reason']


orders_2026-10-18.csv ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'status', 'order_amount', 'promo_value', 'payment_method', 'cancel_reason', 'delivery_fee']
orders_2026-10-19.csv ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'status', 'amount', 'promo_value', 'payment_method', 'cancel_reason']


orders_2026-10-21.csv ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'status', 'amount', 'promo_value', 'payment_method', 'cancel_reason']


## Row count vs distinct order_id per file: spot duplicates

In [3]:
q(f"""SELECT filename, COUNT(*) AS rows, COUNT(DISTINCT order_id) AS distinct_ids
FROM read_csv('{LANDING}/orders/*.csv', all_varchar=true, union_by_name=true, filename=true)
GROUP BY 1 ORDER BY 1""")

filenameVARCHAR,rowsBIGINT,distinct_idsBIGINT
/home/user/data-agent-prac/de-lab/data/landing/orders/orders_2026-10-15.csv,795,795
/home/user/data-agent-prac/de-lab/data/landing/orders/orders_2026-10-16.csv,815,790
/home/user/data-agent-prac/de-lab/data/landing/orders/orders_2026-10-17.csv,831,831
/home/user/data-agent-prac/de-lab/data/landing/orders/orders_2026-10-18.csv,823,823
/home/user/data-agent-prac/de-lab/data/landing/orders/orders_2026-10-19.csv,814,814
/home/user/data-agent-prac/de-lab/data/landing/orders/orders_2026-10-21.csv,826,826


## SUMMARIZE one file
`SUMMARIZE` gives min / max / nulls / distinct counts for every column in one line.

In [4]:
q(f"SUMMARIZE SELECT * FROM read_csv('{LANDING}/orders/orders_2026-10-17.csv')")

column_nameVARCHAR,column_typeVARCHAR,minVARCHAR,maxVARCHAR,approx_uniqueBIGINT,avgVARCHAR,stdVARCHAR,q25VARCHAR,q50VARCHAR,q75VARCHAR,countBIGINT,"null_percentageDECIMAL(9,2)"
order_id,BIGINT,271174,272004,826,271589.0,240.0333310188415,271381,271589,271797,831,0.00
customer_id,BIGINT,13,16517,793,8088.158343483557,4655.826496984046,4132,8100,11973,831,1.20
restaurant_id,BIGINT,1,237,225,117.48495788206979,66.36866333034936,61,117,173,831,0.00
order_ts,VARCHAR,17/10/2026 08:44,2026-10-17 23:27:00,515,NULL,NULL,NULL,NULL,NULL,831,0.00
status,VARCHAR,cancelled,delivered,2,NULL,NULL,NULL,NULL,NULL,831,0.00
amount,DOUBLE,-423.15,1806.0,425,388.8929602888085,199.22044757128523,255.77454545454543,350.4334615384616,474.8222727272727,831,0.00
promo_value,DOUBLE,0.0,100.0,67,22.888086642599276,30.832125947314072,0.0,0.0,50.0,831,0.00
payment_method,VARCHAR,Card,Wallet,4,NULL,NULL,NULL,NULL,NULL,831,0.00
cancel_reason,VARCHAR,customer_cancelled,rider_unavailable,3,NULL,NULL,NULL,NULL,NULL,831,88.81


## Exercises (find them, then ask Claude to write the check)
1. Which file has timestamps in a second format?
2. Which rows have a restaurant_id that does not exist?
3. Which day is missing?
4. Which rows arrived late (order date ≠ file date)?

Answer key: `de-lab/reference/landing_answer_key.md` (no peeking).

In [5]:
# your queries here


## Semi-structured: the nested menu feed (Day 6)

In [6]:
q(f"""SELECT r.restaurant_id, r.name, i.name AS item, i.price.amount AS price_inr, i.tags, i.available
FROM (SELECT unnest(restaurants) AS r FROM read_json('{LANDING}/restaurant_menus.json')),
     LATERAL (SELECT unnest(r.menu.items) AS i)
LIMIT 10""")

restaurant_idBIGINT,nameVARCHAR,itemVARCHAR,price_inrBIGINT,tagsVARCHAR[],availableBOOLEAN
225,Udupi Rasoi,Pav Bhaji,299,[],True
225,Udupi Rasoi,Pani Puri,179,[],True
225,Udupi Rasoi,Vada Pav,129,['veg'],True
153,Spice Grill,Pav Bhaji,129,[],True
153,Spice Grill,Pani Puri,89,[],True
153,Spice Grill,Vada Pav,89,[],True
202,Udupi House,Hakka Noodles,179,"['jain', 'veg']",True
202,Udupi House,Chilli Paneer,129,[],False
202,Udupi House,Manchurian,129,"['jain', 'veg']",True
220,Lucknowi Cafe,Chicken Biryani,249,"['jain', 'bestseller']",True


## Free text: support tickets (Days 16, 18)

In [7]:
q(f"SELECT channel, COUNT(*) FROM read_json('{LANDING}/support_tickets.jsonl') GROUP BY 1")

channelVARCHAR,count_star()BIGINT
whatsapp,99
app,85
email,116
